In [ ]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

# Practice 3 — Evaluation and comparison (R4)

**Scope:** canonical folds, shared metric code, `comparativa.csv` and the paired per-fold differences of P3.5.

## Setup

In [ ]:
import json

import numpy as np
import pandas as pd

import sklearn
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_recall_fscore_support, r2_score)
from sklearn.model_selection import cross_val_predict

# Library versions: some results can change from one version to another
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

### Shared helpers (used by both challenges)

These functions are used in the cyber and in the storage part, so this cell goes into
both final notebooks.

- `canonical_cv` reads the course fold file and returns `folds`, the list of
  `(train_idx, val_idx)` pairs. The folds are read, never rebuilt. As in Practice 2, all the
  folds go through this one function.
- `check_folds` checks the grouping rule: in every fold, no group is both in training and in
  validation, and every row is validated exactly once.
- `mean_and_spread` gives the mean and the spread (max − min) of the five fold values, the
  same definition used in `comparativa.csv`.

In [ ]:
CV_FOLDS = 5   # K, the number of canonical folds


def canonical_cv(challenge, dev):
    """Read the course fold file and return the folds as (train_idx, val_idx) pairs."""
    fold_table = pd.read_csv("../folds/" + challenge + "_folds.csv")

    # The positions only make sense if both files list the rows in the same order
    same_order = np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
    assert same_order, "the fold file and the data table list the rows in a different order"

    fold_id = fold_table["fold_id"].to_numpy()
    folds = []
    for f in range(CV_FOLDS):
        train_idx = np.where(fold_id != f)[0]   # rows of the other four folds
        val_idx = np.where(fold_id == f)[0]     # rows of fold f
        folds.append((train_idx, val_idx))
    return folds

In [ ]:
def check_folds(folds, groups):
    """Check that no group is split between training and validation, fold by fold."""
    groups = np.asarray(groups)
    times_validated = np.zeros(len(groups), dtype=int)

    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        shared = set(groups[train_idx]) & set(groups[val_idx])
        print(f"fold {number}: {len(train_idx)} training rows, {len(val_idx)} validation rows, "
              f"{len(shared)} groups in both parts")
        assert len(shared) == 0, "a group is in training and in validation"
        times_validated[val_idx] = times_validated[val_idx] + 1

    # Every row must be in the validation part of exactly one fold
    assert (times_validated == 1).all(), "a row is not validated exactly once"
    print("every row is validated exactly once")


def mean_and_spread(values):
    """Mean and spread (max - min) of the five fold values."""
    values = np.asarray(values, dtype=float)
    return values.mean(), values.max() - values.min()

### Helpers for `comparativa.csv` and the paired differences

- `append_comparativa_row` adds one row at the end of the table. It never rewrites the rows
  that are already there, and it does not add the same row twice if the notebook is run again.
  It is based on the function of R4 in Practice 2.
- `table_scores` reads the five per-fold values of one row of the table.
- `paired_differences` compares two rows fold by fold. The sign follows the statement
  (Practice 2, Section 5.3): a positive difference always means that the new row did better
  on that fold. For macro F1, $\delta_f = \text{new}_f - \text{P2}_f$; for MAE(z), where lower
  is better, $\delta_f = \text{P2}_f - \text{new}_f$.

The differences are computed from the values as written in the table (four decimals), so
anyone can recompute them from `comparativa.csv`.

In [ ]:
COMPARISON_PATH = "../comparativa.csv"


def fold_text(values):
    """Write the five fold values as one text with 4 decimals, as in comparativa.csv."""
    parts = []
    for value in values:
        parts.append(f"{value:.4f}")
    return " ".join(parts)


def append_comparativa_row(row):
    """Add one row at the end of comparativa.csv. The rows already there are never changed."""
    table = pd.read_csv(COMPARISON_PATH)

    # 1. The new row must have exactly the columns of the table, in the same order
    assert list(row.keys()) == list(table.columns), "the row does not have the table columns"

    # 2. If the same row is already in the table (notebook run again), do not add it twice
    same_row = ((table["milestone"] == row["milestone"])
                & (table["challenge"] == row["challenge"])
                & (table["family"] == row["family"])
                & (table["route"] == row["route"]))
    if same_row.any():
        print("already in comparativa.csv:", row["milestone"], row["challenge"], row["family"])
        return

    # 3. mode="a" writes at the end of the file and leaves every earlier line as it was
    pd.DataFrame([row]).to_csv(COMPARISON_PATH, mode="a", header=False, index=False)
    print("added to comparativa.csv:", row["milestone"], row["challenge"], row["family"])

In [ ]:
def table_scores(milestone, challenge, family):
    """Five per-fold values of the primary metric of one row of comparativa.csv."""
    table = pd.read_csv(COMPARISON_PATH)
    is_row = ((table["milestone"] == milestone)
              & (table["challenge"] == challenge)
              & (table["family"] == family))
    assert is_row.sum() == 1, "expected exactly one row for " + milestone + " " + family
    text = table.loc[is_row, "primary_metric_per_fold"].iloc[0]   # e.g. "0.9436 0.9473 ..."
    return np.array(text.split(), dtype=float)


def paired_differences(challenge, new_scores, old_scores):
    """delta for each fold; a positive delta always means the new row did better."""
    if challenge == "cyber":
        delta = new_scores - old_scores    # macro F1: higher is better
    else:
        delta = old_scores - new_scores    # MAE(z): lower is better

    table = pd.DataFrame({"P2": old_scores, "new": new_scores, "delta": delta},
                         index=range(1, CV_FOLDS + 1))
    table.index.name = "fold"
    print(table.round(4).to_string())

    mean_delta, spread_delta = mean_and_spread(delta)
    print(f"mean delta            {mean_delta:+.4f}")
    print(f"spread of delta       {spread_delta:.4f}   (max - min)")
    print(f"folds with delta > 0  {(delta > 0).sum()} of {CV_FOLDS}")
    print(f"spread of P2 row      {mean_and_spread(old_scores)[1]:.4f}")
    print(f"spread of new row     {mean_and_spread(new_scores)[1]:.4f}")
    return table

**Check of the paired-difference code.** We compare two Practice 2 rows whose differences
Practice 2 already printed: KNN against Dummy in Challenge A, with differences 0.9194,
0.9231, 0.9248, 0.9187 and 0.9209. The values must match up to the four-decimal rounding of
the table (on fold 4 the table gives 0.9186).

In [ ]:
knn_scores = table_scores("P2", "cyber", "KNN")
dummy_scores = table_scores("P2", "cyber", "Dummy")
check = paired_differences("cyber", knn_scores, dummy_scores)

# Each table value is rounded to 4 decimals, so a difference can move by 0.0001
printed_in_p2 = [0.9194, 0.9231, 0.9248, 0.9187, 0.9209]
assert np.allclose(check["delta"], printed_in_p2, atol=0.00015)
print("\nthe function gives the differences printed in Practice 2")

# Challenge A — cyber

> **Only in this draft.** The next cell loads the data with the shared names of the team
> template. In the final notebook these names come from R1's cells, so R5 does not copy it.

In [ ]:
# Only for this draft notebook: in p3_cyber.ipynb these names come from R1's cells
dev = pd.read_csv("../datos/cyber/cyber_development.csv")
groups = pd.read_csv("../datos/cyber/cyber_groups.csv")

NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

X = dev[FEATURES]                       # only the 11 source predictors
y = dev["attack_code"].to_numpy()

# Class code -> class name, e.g. 7 -> "ransomware"
CLASS_NAMES = {}
with open("../datos/cyber/cyber_class_codes.json") as file:
    for code, name in json.load(file).items():
        CLASS_NAMES[int(code)] = name

## Canonical folds and metrics (publish for R2)

The folds are read from `folds/cyber_folds.csv`, the course file. They come from a
`StratifiedGroupKFold(5)` on `feature_group`, so rows with identical predictors always stay in
the same fold.

In [ ]:
# The groups file must list the rows in the same order as the data table
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())

folds = canonical_cv("cyber", dev)

In [ ]:
# Grouping rule: no feature_group may be in training and in validation of the same fold
check_folds(folds, groups["feature_group"])

# X must only hold the source predictors: never row_id, feature_group, config_group or fold_id
assert list(X.columns) == FEATURES

**Metric code for Challenge A** (the metric contract of Practice 2). The model is fitted
outside these functions, for example with `cross_val_predict(model, X, y, cv=folds)`, which
fits a fresh copy of the whole `Pipeline` on the training rows of each fold and predicts its
validation rows. So every row gets one prediction from a model that did not see it, and the
encoder and the scaler are fitted inside each fold.

- `cyber_fold_metrics`: macro F1 (primary), weighted F1 and accuracy on each fold.
- `cyber_class_table`: precision, recall and F1 of each class, beside its support.
- `cyber_confusion`: confusion matrix with the class names (rows = true, columns = predicted).

In [ ]:
def cyber_fold_metrics(y, y_pred, folds):
    """Macro F1 (primary), weighted F1 and accuracy on the validation rows of each fold."""
    y = np.asarray(y)
    rows = []
    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        y_true_fold = y[val_idx]
        y_pred_fold = y_pred[val_idx]
        rows.append({"fold": number,
                     "macro_f1": f1_score(y_true_fold, y_pred_fold, average="macro"),
                     "weighted_f1": f1_score(y_true_fold, y_pred_fold, average="weighted"),
                     "accuracy": accuracy_score(y_true_fold, y_pred_fold)})
    return pd.DataFrame(rows).set_index("fold")

In [ ]:
CLASS_CODES = sorted(CLASS_NAMES)                     # 0, 1, ..., 9
CLASS_LABELS = []
for class_code in CLASS_CODES:
    CLASS_LABELS.append(CLASS_NAMES[class_code])      # "backdoor", "ddos", ...


def cyber_class_table(y, y_pred):
    """Precision, recall and F1 of each class, beside its support (all folds together)."""
    precision, recall, f1, support = precision_recall_fscore_support(
        y, y_pred, labels=CLASS_CODES, zero_division=0)
    return pd.DataFrame({"precision": precision, "recall": recall, "f1": f1,
                         "support": support}, index=CLASS_LABELS)


def cyber_confusion(y, y_pred):
    """Confusion matrix with class names: rows = true class, columns = predicted class."""
    matrix = confusion_matrix(y, y_pred, labels=CLASS_CODES)
    table = pd.DataFrame(matrix, index=CLASS_LABELS, columns=CLASS_LABELS)
    table.index.name = "true"
    table.columns.name = "predicted"
    return table

**Check: the folds and the metric code reproduce Practice 2.** We run again the Practice 2
baseline, `DummyClassifier(strategy="most_frequent")`, and compare its macro F1 per fold
with the P2 row of `comparativa.csv`. If the folds or the metric code had changed, the
values would not match. The Dummy model does not use `X`, so it needs no preprocessing.

In [ ]:
y_pred_dummy = cross_val_predict(DummyClassifier(strategy="most_frequent"), X, y, cv=folds)
dummy_metrics = cyber_fold_metrics(y, y_pred_dummy, folds)
print(dummy_metrics.round(4).to_string())

p2_dummy = table_scores("P2", "cyber", "Dummy")
assert np.allclose(dummy_metrics["macro_f1"], p2_dummy, atol=0.00005)
print("\nmacro F1 per fold equals the P2 Dummy row of comparativa.csv")

## Paired comparison against Practice 2

In [ ]:
# TODO (when R2's results are final): build the P3 cyber row, from R2's saved outputs,
# with the columns of comparativa.csv and route = "primary", and add it with
# append_comparativa_row(row). Then:
#     new_scores = table_scores("P3", "cyber", "<family of the P3 row>")
#     old_scores = table_scores("P2", "cyber", "KNN")
#     cyber_deltas = paired_differences("cyber", new_scores, old_scores)

**Reading of the differences, and which kind of comparison this is.** TODO

# Challenge B — storage

> **Only in this draft.** As in Challenge A, in the final notebook these names come from
> R1's cells, so R5 does not copy the next cell.

In [ ]:
# Only for this draft notebook: in p3_storage.ipynb these names come from R1's cells
dev = pd.read_csv("../datos/storage/storage_development.csv")
groups = pd.read_csv("../datos/storage/storage_groups.csv")

NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

X = dev[FEATURES]                         # only the 9 source predictors
latency_ns = dev["latency_ns"].to_numpy()
z = np.log10(latency_ns)                  # modelling target, z = log10(latency_ns)

## Canonical folds and metrics (publish for R3)

The folds are read from `folds/storage_folds.csv`, the course file. They come from a
`GroupKFold(5)` on `config_group`, so the five replicates of a configuration always stay in
the same fold.

In [ ]:
# The groups file must list the rows in the same order as the data table
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())

folds = canonical_cv("storage", dev)

In [ ]:
# Grouping rule: no config_group may be in training and in validation of the same fold
check_folds(folds, groups["config_group"])

# X must only hold the source predictors: never row_id, feature_group, config_group or fold_id
assert list(X.columns) == FEATURES

**Metric code for Challenge B** (the metric contract of Practice 2). As in Challenge A, the
model is fitted outside the function, for example with `cross_val_predict(model, X, z, cv=folds)`,
so the prediction of each row comes from a model that did not see it.

- MAE(z) (primary), RMSE(z) and R²(z) are computed from $z$ and $\hat z$.
- MAE in ns is computed only after turning each prediction back into nanoseconds with
  $10^{\hat z}$. Raising 10 to MAE(z) gives a multiplicative factor, never an error in ns.

In [ ]:
def storage_fold_metrics(z, z_pred, latency_ns, folds):
    """MAE(z), RMSE(z), R2(z) and MAE in ns on the validation rows of each fold."""
    rows = []
    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        z_true_fold = z[val_idx]
        z_pred_fold = z_pred[val_idx]
        # Back to nanoseconds: we invert the predictions, never the MAE
        latency_pred_fold = 10 ** z_pred_fold
        rows.append({"fold": number,
                     "mae_z": mean_absolute_error(z_true_fold, z_pred_fold),
                     "rmse_z": np.sqrt(mean_squared_error(z_true_fold, z_pred_fold)),
                     "r2_z": r2_score(z_true_fold, z_pred_fold),
                     "mae_ns": mean_absolute_error(latency_ns[val_idx], latency_pred_fold)})
    return pd.DataFrame(rows).set_index("fold")

**Check: the folds and the metric code reproduce Practice 2.** We run again the Practice 2
baseline, `DummyRegressor(strategy="mean")` on $z$, and compare its MAE(z) per fold with the
P2 row of `comparativa.csv`.

In [ ]:
z_pred_dummy = cross_val_predict(DummyRegressor(strategy="mean"), X, z, cv=folds)
dummy_metrics = storage_fold_metrics(z, z_pred_dummy, latency_ns, folds)
print(dummy_metrics.round(4).to_string())

p2_dummy = table_scores("P2", "storage", "Dummy")
assert np.allclose(dummy_metrics["mae_z"], p2_dummy, atol=0.00005)
print("\nMAE(z) per fold equals the P2 Dummy row of comparativa.csv")

## Paired comparison against Practice 2

In [ ]:
# TODO (when R3's results are final): build the P3 storage row(s), from R3's saved outputs,
# with the columns of comparativa.csv and route = "primary", and add them with
# append_comparativa_row(row). Then:
#     new_scores = table_scores("P3", "storage", "<family of the P3 row>")
#     old_scores = table_scores("P2", "storage", "KNN")
#     storage_deltas = paired_differences("storage", new_scores, old_scores)

**Reading of the differences, and which kind of comparison this is.** TODO

# Comparison table

The Practice 3 rows are added in the paired-comparison part of each challenge, just before
their differences are computed, because the differences are read from the table. This last
cell shows the whole table so that we can check that the four Practice 2 rows are still the
first rows, unchanged, and that the new rows come after them.

Practice 2 and Practice 3 do not use the same representation of the nominal predictors:
Practice 2 one-hot encodes every level, Practice 3 drops one reference level per predictor.
The information basis is the same (the same source predictors, no derived feature), but the
representation is not, so every Practice 3 against Practice 2 difference is a **pipeline
comparison**, not an isolated comparison of model families.

In [ ]:
table = pd.read_csv(COMPARISON_PATH)
print(table[["milestone", "challenge", "family", "route",
             "primary_metric", "primary_metric_spread"]].to_string())

# The table is only appended to: the Practice 2 rows are still the first four rows
assert list(table["milestone"].iloc[:4]) == ["P2", "P2", "P2", "P2"]